# 1. Import of input data

In [1]:
import nbimporter
import sys
import os
from pathlib import Path

In [2]:
#Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 ThermalConcreteStorage.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:
    current_dir = Path().resolve()
    print(f"📁 ThermalConcreteStorage.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries_pd.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries_pd.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importing libraries from: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries_pd.ipynb not found! Some functions may not be available.")

📁 ThermalConcreteStorage.ipynb executed standalone — using directory: C:\Users\Saverio_admin\Documents\NEST_py\Energy_systems_pd
📘 Importing libraries from: C:\Users\Saverio_admin\Documents\NEST_py\Libraries_pd.ipynb
📁 Libraries_pd.ipynb — directory impostata a: C:\Users\Saverio_admin\Documents\NEST_py


In [3]:
energysystem = pd.read_excel(energysystem_path, sheet_name = 'Concrete thermal storage')

# 2. Input parameters

In [ ]:
ConcreteStorage_BdName = energysystem['Building Name']
ConcreteStorage_BdArea = energysystem['Zone net floor area [m2]']
VolumeStored = energysystem['Storage [m3]'];
EnergyStored = energysystem['Energy stored [kWh/year]'];

In [5]:
Lifespan_Storage = 40
RefVolume = 10000 * 0.1 #cubic meters
volume = VolumeStored
x = RefVolume / volume
mass = 2400 * x

# 3. Life cycle inventories

## 3.1 Module A1-3

In [6]:
ThermalConcreteStorage_steellowalloyed = 0.37 * mass
ThermalConcreteStorage_steelunalloyed = 0.63 * mass
ThermalConcreteStorage_concrete = 0.98
ThermalConcreteStorage_stonewool = 0.00219 * mass 

## 3.2 Module A4

In [7]:
ThermalConcreteStorage_Truk_16_32 = ((500 + 200)*(mass)/1000)
ThermalConcreteStorage_Light_Truk = (50*(mass)/1000)

## 3.3 Module A5

In [8]:
ThermalConcreteStorage_excavation = x

## 3.4 Module B2

In [9]:
ThermalConcreteStorage_Light_Truk_B2 = (50 * (Lifespan_Storage/2) * 0.01 * (mass/1000))

## 3.5 Module B4

In [10]:
ThermalConcreteStorageNumberReplacements = (np.ceil(np.divide(RSP, Lifespan_Storage) - 1))
SFThermalConcreteStorage = (RSP / Lifespan_Storage)  # Scaling factor

## 3.6 Module C2

In [11]:
ThermalConcreteStorage_Transport_to_treatment_plant = 50 * (mass/1000)

## 3.7 Module C3

In [12]:
ThermalConcreteStorage_EOL_steellowalloyed = ThermalConcreteStorage_steellowalloyed
ThermalConcreteStorage_EOL_steelunalloyed = ThermalConcreteStorage_steelunalloyed
ThermalConcreteStorage_EOL_concrete = ThermalConcreteStorage_concrete
ThermalConcreteStorage_EOL_stonewool = ThermalConcreteStorage_stonewool

## 3.8 Module C4

In [13]:
ThermalConcreteStorage_EOL_landfill_steellowalloyed = ThermalConcreteStorage_EOL_steellowalloyed * LandfillMetals
ThermalConcreteStorage_EOL_landfill_Steelunalloyed = ThermalConcreteStorage_EOL_steelunalloyed * LandfillMetals
ThermalConcreteStorage_EOL_landfill_concrete = ThermalConcreteStorage_EOL_concrete * LandfillConcrete
ThermalConcreteStorage_EOL_landfill_stonewool = ThermalConcreteStorage_EOL_stonewool * LandfillMineralWool

## 3.9 Module D1 (related to the export of secondary materials)

In [14]:
ThermalConcreteStorage_EOL_recycling_steellowalloyed = (RecyclingMetals * RecyclingEfficiencyMetals * ThermalConcreteStorage_EOL_steellowalloyed) - (RecycledContentMetals * ThermalConcreteStorage_EOL_steellowalloyed)
ThermalConcreteStorage_EOL_primary_steellowalloyed = ((RecycledContentMetals * ThermalConcreteStorage_EOL_steellowalloyed) - (RecyclingMetals * RecyclingEfficiencyMetals * ThermalConcreteStorage_EOL_steellowalloyed))* SRmetals

ThermalConcreteStorage_EOL_recycling_Steelunalloyed = (RecyclingMetals * RecyclingEfficiencyMetals * ThermalConcreteStorage_EOL_steelunalloyed) - (RecycledContentMetals * ThermalConcreteStorage_EOL_steelunalloyed)
ThermalConcreteStorage_EOL_primary_Steelunalloyed = ((RecycledContentMetals * ThermalConcreteStorage_EOL_steelunalloyed) - (RecyclingMetals * RecyclingEfficiencyMetals * ThermalConcreteStorage_EOL_steelunalloyed))* SRmetals

ThermalConcreteStorage_EOL_recycling_concrete = (RecyclingConcrete * RecyclingEfficiencyConcrete * ThermalConcreteStorage_EOL_concrete) - (RecycledContentConcrete * ThermalConcreteStorage_EOL_concrete)
ThermalConcreteStorage_EOL_primary_concrete = ((RecycledContentConcrete * ThermalConcreteStorage_EOL_concrete) - (RecyclingConcrete * RecyclingEfficiencyConcrete * ThermalConcreteStorage_EOL_concrete)) * SRinert


# 4. Data for export